<table style="width: 100%; border-collapse: collapse; border: none; background: #f8fafc; border-left: 6px solid #1e3a8a; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #0f172a; font-size: 2.1em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        DBSCAN y Clustering Basado en Densidad
      </h1>
      <p style="margin: 6px 0 0 0; color: #1e3a8a; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Programación para Ciencia de Datos
      </p>
      <p style="margin: 4px 0 0 0; color: #64748b; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #1e3a8a; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        Módulo 10
      </span><br>
      <span style="color: #64748b; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #2563eb; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Data%20Science%20programming/10%20-%20Clustering/03_DBSCAN_y_Clustering_Basado_en_Densidad.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os, urllib.request
import warnings
warnings.filterwarnings('ignore')

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (8.5, 4.5)
plt.rcParams['font.size'] = 10

def load_dataset(filename, module_folder="10 - Clustering"):
    local_path = os.path.join(os.getcwd(), "data", filename)
    if os.path.exists(local_path):
        return local_path
    
    parent_path = os.path.join(os.getcwd(), "..", module_folder, "data", filename)
    if os.path.exists(parent_path):
        return parent_path

    raw_url = f"https://raw.githubusercontent.com/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/main/Data%20Science%20programming/{module_folder.replace(' ', '%20')}/data/{filename}"
    os.makedirs("data", exist_ok=True)
    target_path = os.path.join("data", filename)
    if not os.path.exists(target_path):
        urllib.request.urlretrieve(raw_url, target_path)
    return target_path

print("🚀 Entorno configurado exitosamente para el Módulo 10: Clustering.")


---
### 1. ¿Por qué Clustering Basado en Densidad? (Limitaciones de K-Means) 🌧️

Tanto **K-Means** como **HAC (Ward)** asumen que los clusters son aproximadamente **esféricos, convexos y libres de ruido extremo**. Sin embargo, en el mundo real encontramos patrones con **formas arbitrarias (anillos, semilunas, densidades variables) y datos atípicos**.

**DBSCAN (*Density-Based Spatial Clustering of Applications with Noise*)** descubre clusters como regiones densas continuas separadas por regiones de baja densidad.

<div align="center">
  <img src="images/dbscan.png" width="480" alt="DBSCAN Point Types" style="border-radius: 8px; box-shadow: 0 4px 6px rgba(0,0,0,0.1); margin: 10px 0;"/>
</div>


---
### 2. Fundamentos Matemáticos y Tipos de Puntos en DBSCAN 🔍

DBSCAN requiere dos hiperparámetros fundamentales:
1. **$arepsilon$ (`eps`):** Radio máximo de la vecindad alrededor de un punto $p$.
   $$N_arepsilon(p) = \{ q \in D \mid \text{dist}(p, q) \le arepsilon \}$$
2. **`min_samples` ($MinPts$):** Número mínimo de puntos requeridos dentro de la $arepsilon$-vecindad para que $p$ sea considerado un punto denso.

#### Clasificación de Puntos:
* **Punto Núcleo (*Core Point*):** $|N_arepsilon(p)| \ge \text{min\_samples}$.
* **Punto de Borde (*Border Point*):** No es núcleo ($|N_arepsilon(q)| < \text{min\_samples}$), pero pertenece a la vecindad de un punto núcleo ($q \in N_arepsilon(p)$).
* **Punto de Ruido (*Noise / Outlier*):** No es núcleo ni pertenece a la vecindad de ningún núcleo. Scikit-Learn le asigna la etiqueta `-1`.


In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.datasets import make_moons, make_circles

# Crear dataset no convexo (dos semilunas entrelazadas)
X_moons, _ = make_moons(n_samples=300, noise=0.08, random_state=42)

# Ajustar DBSCAN
dbscan = DBSCAN(eps=0.2, min_samples=5)
labels_moons = dbscan.fit_predict(X_moons)

n_clusters_ = len(set(labels_moons)) - (1 if -1 in labels_moons else 0)
n_noise_ = list(labels_moons).count(-1)

print(f"Clusters descubiertos automáticamente: {n_clusters_}")
print(f"Puntos clasificados como ruido (outliers): {n_noise_}")


---
### 3. Comparativa Visual: K-Means vs DBSCAN en Geometrías Complejas 🌓


In [ ]:
from sklearn.cluster import KMeans

# Ajustar KMeans en el mismo dataset
kmeans_moons = KMeans(n_clusters=2, random_state=42).fit_predict(X_moons)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# K-Means
axes[0].scatter(X_moons[:, 0], X_moons[:, 1], c=kmeans_moons, cmap='coolwarm', s=45, alpha=0.8)
axes[0].set_title('K-Means (k=2) — Falla en Formas No Convexas', fontweight='bold', color='#dc2626')

# DBSCAN
axes[1].scatter(X_moons[labels_moons != -1, 0], X_moons[labels_moons != -1, 1], 
                c=labels_moons[labels_moons != -1], cmap='viridis', s=45, alpha=0.8, label='Clusters')
axes[1].scatter(X_moons[labels_moons == -1, 0], X_moons[labels_moons == -1, 1], 
                c='red', marker='x', s=60, label='Ruido (-1)')
axes[1].set_title('DBSCAN (eps=0.2, min_samples=5) — Captura Perfecta', fontweight='bold', color='#16a34a')
axes[1].legend()

plt.tight_layout()
plt.show()


---
### 4. Heurística para Determinar $arepsilon$ Óptimo (Gráfico de $k$-Distancias) 📈

Para seleccionar $arepsilon$, calculamos la distancia de cada punto a su $k$-ésimo vecino más cercano ($k = \text{min\_samples}$) y graficamos las distancias ordenadas. El punto de **máxima curvatura ("codo")** indica el valor de $arepsilon$ ideal:


In [ ]:
from sklearn.neighbors import NearestNeighbors

# Gráfico de k-distancias sobre datos de clientes
path_data = load_dataset('mall_customers.csv', '10 - Clustering')
df = pd.read_csv(path_data)
X_cust = df[['Annual_Income_k', 'Spending_Score']].values
X_cust_scaled = StandardScaler().fit_transform(X_cust)

k_neighbors = 5
nn = NearestNeighbors(n_neighbors=k_neighbors)
nn.fit(X_cust_scaled)
distancias, _ = nn.kneighbors(X_cust_scaled)

# Ordenar las distancias al k-ésimo vecino
k_distances = np.sort(distancias[:, k_neighbors - 1])

plt.figure(figsize=(8.5, 4.2))
plt.plot(k_distances, color='#0284c7', linewidth=2)
plt.axhline(y=0.35, color='r', linestyle='--', label='Punto de Inflexión sugerido eps ≈ 0.35')
plt.title(f'Gráfico de {k_neighbors}-Distancias para Selección de eps', fontweight='bold')
plt.xlabel('Observaciones Ordenadas por Distancia')
plt.ylabel(f'Distancia al {k_neighbors}° Vecino')
plt.legend()
plt.tight_layout()
plt.show()


---
##### 🛠️ Práctica 4: Detección de Anomalías en Clientes con DBSCAN

**Reto:**
1. Ajusta un modelo `DBSCAN(eps=0.35, min_samples=5)` sobre `X_cust_scaled`.
2. Identifica el número de clusters detectados y la cantidad de clientes etiquetados como ruido (`-1`).
3. Grafica los clusters descubiertos y resalta los clientes anómalos en color rojo con marcador `'x'`.
4. Examina los valores en escala original de los clientes ruidosos para entender por qué son anomalías.


In [ ]:
# =========================================================================
# TU SOLUCIÓN: Práctica 4 - DBSCAN y Detección de Anomalías
# =========================================================================

# 1. Ajustar DBSCAN
# db = DBSCAN(eps=0.35, min_samples=5)
# db_labels = db.fit_predict(X_cust_scaled)

# 2. Conteo de clusters y ruido
# ...


<details>
<summary><b>💡 Haz clic aquí para ver la Solución Paso a Paso y Explicación</b></summary>
<br>

```python
# 1. Ajustar modelo DBSCAN
db = DBSCAN(eps=0.35, min_samples=5)
db_labels = db.fit_predict(X_cust_scaled)

n_c = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_r = (db_labels == -1).sum()

print(f"Clusters identificados: {n_c}")
print(f"Clientes anómalos (Ruido): {n_r}")

# 3. Graficar
plt.figure(figsize=(9, 4.5))
plt.scatter(X_cust[db_labels != -1, 0], X_cust[db_labels != -1, 1], 
            c=db_labels[db_labels != -1], cmap='tab10', s=55, alpha=0.8, label='Clientes Normales')
plt.scatter(X_cust[db_labels == -1, 0], X_cust[db_labels == -1, 1], 
            c='red', marker='x', s=90, linewidths=2, label='Anomalías / Ruido')

plt.title('DBSCAN en Segmentación de Clientes', fontweight='bold')
plt.xlabel('Ingreso Anual (k USD)')
plt.ylabel('Puntuación de Gasto (1-100)')
plt.legend()
plt.tight_layout()
plt.show()

# 4. Ver clientes anómalos
print("\nClientes anómalos identificados:\n", df[db_labels == -1][['Age', 'Annual_Income_k', 'Spending_Score']])
```
</details>


---
### 5. Resumen y Conclusiones del Cuaderno 03 📌

1. **Formas Arbitrarias:** DBSCAN no impone supuestos de convexidad ni esfericidad, siendo capaz de descubrir clusters con formas complejas.
2. **Detección Automática de Ruido:** Los valores atípicos se aíslan naturalmente con la etiqueta `-1` sin distorsionar los centroides.
3. **Sensibilidad a Parámetros:** Si $arepsilon$ es muy pequeño, todo será ruido; si es muy grande, todos los clusters se fusionarán en uno solo.

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Programación para Ciencia de Datos</i>
  </p>
</div>
